In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("sdac2018processed.csv")
print("Loaded dataset:", df.shape)

Loaded dataset: (9271, 112)


In [3]:
predictor_vars = [
    "AGEPC", "SEX", "EDLVLATB", "INCDECHD", "ARIAC", "SPENGPRO",
    "HHNOPSNB", "SOCAWAYCOUNT", "SOCHOMECOUNT", "SOCPARTICIPATIONTYPE",
    "SOCMORCO", "K10CATEG", "DISBSTAT", "SFHEALTH"
]

need_vars = [
    "WHNASMOB", "WHNASSCR", "WHNASCOM", "NAHCFCAR", "WHNASEMO",
    "WHNASHOM", "WHNASPRO", "WNASMEAL", "WHNASFIN", "NASTRANS"
]

informal_vars = [
    "RASFRICO", "RASFRIMO", "RASFRISC", "RASFRIEM", "RASFRIHC",
    "RASFRIHO", "RASFRIME", "RASFRIPM", "RASFRIPA", "RASFRITR"
]

outcome_vars = [
    "RASEXMOB", "RASEXSC", "RASEXCOM", "RASEXHC", "RASEXGUI",
    "RASEXHOM", "RASEXPRP", "RASEXMEA", "RASEXPAP", "RASEXTRA"
]

domain_mapping = {
    "Mobility": ("WHNASMOB", "RASEXMOB"),
    "Self-care": ("WHNASSCR", "RASEXSC"),
    "Communication": ("WHNASCOM", "RASEXCOM"),
    "Health care": ("NAHCFCAR", "RASEXHC"),
    "Cognition/emotion": ("WHNASEMO", "RASEXGUI"),
    "Household chores": ("WHNASHOM", "RASEXHOM"),
    "Property maintenance": ("WHNASPRO", "RASEXPRP"),
    "Meal preparation": ("WNASMEAL", "RASEXMEA"),
    "Reading/writing": ("WHNASFIN", "RASEXPAP"),
    "Transport": ("NASTRANS", "RASEXTRA")
}

In [4]:
non_health_need_vars = [x for x in need_vars if x != "NAHCFCAR"]

has_any_need = (
    df[non_health_need_vars].eq(1).any(axis=1)
    | df["NAHCFCAR"].isin([1, 2, 3])
)

df = df.loc[has_any_need].copy()

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("Duplicate rows:", df.duplicated().sum())

Rows: 3270
Columns: 112
Duplicate rows: 0


In [5]:
print("shape", df.shape)
print("rows", df.shape[0])
print("columns", df.shape[1])
print("duplicate", df.duplicated().sum())


shape (3270, 112)
rows 3270
columns 112
duplicate 0


In [6]:
missing = pd.DataFrame({
    "Variable": df.columns,
    "Missing_Count": df.isna().sum().values,
    "Missing_Percent": (df.isna().mean().values * 100).round(2)
}).sort_values("Missing_Percent", ascending=False)

missing

,Variable,Missing_Count,Missing_Percent
0,ABSPID,0,0.0
1,ABSFID,0,0.0
2,ABSHID,0,0.0
3,ABSIID,0,0.0
4,POPESTAB,0,0.0
...,...,...,...
107,RASEXHOM,0,0.0
108,RASEXPRP,0,0.0
109,RASEXMEA,0,0.0
110,RASEXPAP,0,0.0


In [9]:
eda_vars = (
    predictor_vars
    + need_vars
    + informal_vars
    + outcome_vars
)
def variable_counts(df, variables):
    results = []

    for col in variables:

        counts = (
            df[col]
            .value_counts(dropna=False)
            .sort_index()
        )

        percentages = (
            df[col]
            .value_counts(
                dropna=False,
                normalize=True
            )
            .sort_index()
            .mul(100)
            .round(2)
        )

        temp = pd.DataFrame({
            "Variable": col,
            "Code": counts.index,
            "Count": counts.values,
            "Percentage": percentages.values
        })

        results.append(temp)

    return pd.concat(
        results,
        ignore_index=True
    )
all_counts = variable_counts(
    df,
    eda_vars
)

all_counts

,Variable,Code,Count,Percentage
0,AGEPC,25,611,18.69
1,AGEPC,26,704,21.53
2,AGEPC,27,658,20.12
3,AGEPC,28,592,18.10
4,AGEPC,29,705,21.56
...,...,...,...,...
259,RASEXPAP,3,16,0.49
260,RASEXTRA,0,1841,56.30
261,RASEXTRA,1,1244,38.04
262,RASEXTRA,2,125,3.82


In [10]:
need_indicators = pd.DataFrame(index=df.index)

for col in need_vars:

    if col == "NAHCFCAR":
        # Health care: codes 1, 2, 3 = has assistance need
        need_indicators[col] = df[col].isin([1, 2, 3]).astype(int)

    else:
        # Other domains: code 1 = has assistance need
        need_indicators[col] = df[col].eq(1).astype(int)
        
df["NUMBER_OF_NEED_DOMAINS"] = (
    need_indicators
    .sum(axis=1)
    .astype(int)
)
counts = (
    df["NUMBER_OF_NEED_DOMAINS"]
    .value_counts()
    .reindex(range(1, 11), fill_value=0)
)

percentages = (
    counts / len(df) * 100
).round(2)

need_domain_summary = pd.DataFrame({
    "Number_of_Domains": counts.index,
    "Count": counts.values,
    "Percentage": percentages.values
})

need_domain_summary

,Number_of_Domains,Count,Percentage
0,1,1065,32.57
1,2,656,20.06
2,3,452,13.82
3,4,306,9.36
4,5,272,8.32
5,6,186,5.69
6,7,145,4.43
7,8,103,3.15
8,9,65,1.99
9,10,20,0.61


In [11]:
domain_unmet_summary = []

for domain, (need_var, outcome_var) in domain_mapping.items():

    # Define who has a need
    if need_var == "NAHCFCAR":
        has_need = df[need_var].isin([1, 2, 3])
    else:
        has_need = df[need_var].eq(1)

    domain_df = df.loc[has_need]

    total_need = len(domain_df)

    unmet_count = domain_df[outcome_var].isin([2, 3]).sum()

    unmet_percent = (
        unmet_count / total_need * 100
        if total_need > 0 else np.nan
    )

    domain_unmet_summary.append({
        "Domain": domain,
        "People_Needing_Assistance": total_need,
        "Unmet_Count": unmet_count,
        "Unmet_Percentage": round(unmet_percent, 2)
    })

domain_unmet_summary = pd.DataFrame(domain_unmet_summary)

domain_unmet_summary

,Domain,People_Needing_Assistance,Unmet_Count,Unmet_Percentage
0,Mobility,1066,161,15.10
1,Self-care,673,76,11.29
2,Communication,144,15,10.42
3,Health care,1787,252,14.10
4,Cognition/emotion,495,99,20.00
5,Household chores,1572,339,21.56
6,Property maintenance,1981,589,29.73
7,Meal preparation,459,52,11.33
8,Reading/writing,451,35,7.76
9,Transport,1429,185,12.95


In [ ]:
for domain, (need_var, outcome_var) in domain_mapping.items():
    if need_var == "NAHCFCAR":
        has_need = df[need_var].isin([1, 2, 3])
    else:
        has_need = df[need_var] == 1

    error_1 = (
        (~has_need) &
        (df[outcome_var] != 0)
    ).sum()

    error_2 = (
        (has_need) &
        (df[outcome_var] == 0)
    ).sum()

    print(f"{domain}")
    print("No need but RASEX is not 0:", error_1)
    print("Has need but RASEX is 0:", error_2)

Mobility
No need but RASEX != 0: 0
Has need but RASEX = 0: 0
Self-care
No need but RASEX != 0: 0
Has need but RASEX = 0: 0
Communication
No need but RASEX != 0: 0
Has need but RASEX = 0: 0
Health care
No need but RASEX != 0: 0
Has need but RASEX = 0: 0
Cognition/emotion
No need but RASEX != 0: 0
Has need but RASEX = 0: 0
Household chores
No need but RASEX != 0: 0
Has need but RASEX = 0: 0
Property maintenance
No need but RASEX != 0: 0
Has need but RASEX = 0: 0
Meal preparation
No need but RASEX != 0: 0
Has need but RASEX = 0: 0
Reading/writing
No need but RASEX != 0: 0
Has need but RASEX = 0: 0
Transport
No need but RASEX != 0: 0
Has need but RASEX = 0: 0


In [13]:
domain_outcome_breakdown = []

for domain, (need_var, outcome_var) in domain_mapping.items():

    # Define who needs assistance
    if need_var == "NAHCFCAR":
        has_need = df[need_var].isin([1, 2, 3])
    else:
        has_need = df[need_var].eq(1)

    domain_df = df.loc[has_need]

    total = len(domain_df)

    for code in [1, 2, 3]:

        count = (domain_df[outcome_var] == code).sum()

        percentage = (
            count / total * 100
            if total > 0 else np.nan
        )

        domain_outcome_breakdown.append({
            "Domain": domain,
            "Outcome_Code": code,
            "Count": count,
            "Percentage": round(percentage, 2)
        })

domain_outcome_breakdown = pd.DataFrame(
    domain_outcome_breakdown
)

domain_outcome_breakdown

,Domain,Outcome_Code,Count,Percentage
0,Mobility,1,905,84.90
1,Mobility,2,126,11.82
2,Mobility,3,35,3.28
3,Self-care,1,597,88.71
4,Self-care,2,48,7.13
5,Self-care,3,28,4.16
6,Communication,1,129,89.58
7,Communication,2,13,9.03
8,Communication,3,2,1.39
9,Health care,1,1535,85.90


In [ ]:
with pd.ExcelWriter(
    "SDAC_EDA_summary.xlsx",
    engine="openpyxl"
) as writer:

    all_counts.to_excel(
        writer,
        sheet_name="Variable Counts",
        index=False
    )

    need_domain_summary.to_excel(
        writer,
        sheet_name="Need Domains",
        index=False
    )

    domain_outcome_breakdown.to_excel(
        writer,
        sheet_name="Domain Outcomes",
        index=False
    )

print("Saved to SDAC_EDA_summary.xlsx")